# BPR — régimen **F2** sobre **SCGRec**

Full-ranking · split random 80/10/10 (re-split de inter_train, protocolo CPGRec) · métricas `Recall/NDCG/Hit/Precision@5,10` + `Cov/Ent` + long-tail · **1 seed** (Colab gratuito).

> Bootstrap autocontenido (gdown + parse, = `load_scgrec`). **Re-split RANDOM 80/10/10 de `inter_train`** (protocolo CPGRec §5.1.1, = corregido T2; NO el split oficial valid/test). Positivo implícito; ~40% playtime=0.

> Plantilla = `deep_kozyriev_h3.ipynb`; modelo enchufado al harness F2 importado de `recsys_protocol.py`. Las filas MostPop/ALS son **self-check** (deben reproducir las del `*_corregido`).

In [1]:
# ====== Bootstrap: deps + clonar repo + cargar recsys_protocol ======
!pip install -q kagglehub implicit
import os, sys, glob, json, math, time, csv, zipfile, shutil, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

REPO_URL = 'https://github.com/Benjaa7/Proyecto-RecSys.git'
REPO_DIR = '/content/Proyecto-RecSys'
def _locate_protocol():
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git','-C',REPO_DIR,'fetch','-q','--depth','1','origin'], check=False)
        subprocess.run(['git','-C',REPO_DIR,'reset','--hard','-q','FETCH_HEAD'], check=False)
    else:
        subprocess.run(['git','clone','--depth','1','-q',REPO_URL,REPO_DIR], check=False)
    h3 = os.path.join(REPO_DIR, 'H3')
    if os.path.exists(os.path.join(h3, 'recsys_protocol.py')):
        return h3
    for c in ['/content','.','..','H3','../H3'] + sorted(glob.glob('/content/drive/MyDrive/*')):
        if c and os.path.exists(os.path.join(c, 'recsys_protocol.py')):
            return c
    return None
_p = _locate_protocol()
assert _p, 'No encontre recsys_protocol.py (clona el repo o sube el modulo).'
if _p not in sys.path: sys.path.insert(0, _p)

from recsys_protocol import SEED, set_global_seed, iterative_k_core
set_global_seed()
# Sección F2: usar la del módulo si está publicada; si el repo clonado trae una
# versión vieja de recsys_protocol (sin F2), caer a definiciones inline IDÉNTICAS.
try:
    from recsys_protocol import random_split_8010, paper_metrics, cat_cov_ent, recs_from_embeddings, longtail_ndcg
    print('[F2] funciones importadas de recsys_protocol')
except ImportError:
    print('[F2] recsys_protocol clonado sin sección F2 -> usando definiciones inline (idénticas al módulo)')
    def random_split_8010(inter, seed=SEED):
        inter = inter.reset_index(drop=True); rng = np.random.default_rng(seed); n = len(inter)
        inter = inter.iloc[rng.permutation(n)].reset_index(drop=True)
        n_tr, n_va = int(0.8 * n), int(0.1 * n)
        sp = np.empty(n, dtype='int8'); sp[:n_tr] = 0; sp[n_tr:n_tr + n_va] = 1; sp[n_tr + n_va:] = 2
        inter['split'] = sp; return inter
    def _dcg_f2(hits): return sum((1.0 / math.log2(i + 2)) for i, h in enumerate(hits) if h)
    def cat_cov_ent(recs, cat_map, k):
        covs, ents = [], []
        for rec in recs.values():
            cnt = {}
            for it in rec[:k]:
                for c in cat_map.get(it, ()): cnt[c] = cnt.get(c, 0) + 1
            if not cnt: covs.append(0); ents.append(0.0); continue
            covs.append(len(cnt)); tot = sum(cnt.values())
            ents.append(-sum((v / tot) * math.log2(v / tot) for v in cnt.values()))
        return (float(np.mean(covs)) if covs else 0.0, float(np.mean(ents)) if ents else 0.0)
    def paper_metrics(recs, test_items, ks=(5, 10), cat_maps=None):
        acc = {f'{m}@{k}': [] for k in ks for m in ('Recall', 'NDCG', 'Hit', 'Precision')}; n = 0
        for u, rec in recs.items():
            rel = test_items.get(u)
            if not rel: continue
            n += 1
            for k in ks:
                hits = [(1 if it in rel else 0) for it in rec[:k]]; nhit = sum(hits)
                acc[f'Recall@{k}'].append(nhit / len(rel)); acc[f'Precision@{k}'].append(nhit / k)
                acc[f'Hit@{k}'].append(1.0 if nhit > 0 else 0.0)
                idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(rel), k)))
                acc[f'NDCG@{k}'].append(_dcg_f2(hits) / idcg if idcg > 0 else 0.0)
        out = {key: (float(np.mean(v)) if v else 0.0) for key, v in acc.items()}
        if cat_maps:
            for k in ks:
                for name, cmap in cat_maps.items():
                    cov, ent = cat_cov_ent(recs, cmap, k); out[f'Cov_{name}@{k}'] = cov; out[f'Ent_{name}@{k}'] = ent
        out['n_users'] = n; return out
    def recs_from_embeddings(e_u, e_i, umap, idx2app, users, topn, train_items_per_user, popular_list):
        out = {}; nfb = 0
        for u in users:
            seen = train_items_per_user.get(u, set()); key = str(u)
            if key not in umap:
                out[u] = [i for i in popular_list if i not in seen][:topn]; nfb += 1; continue
            scores = e_i @ e_u[umap[key]]; rec = []
            for j in np.argsort(-scores):
                a = idx2app[int(j)]
                if a not in seen:
                    rec.append(a)
                    if len(rec) >= topn: break
            out[u] = rec
        return out, nfb
    def _bucket_f2(n): return '2-5' if n <= 5 else '6-20' if n <= 20 else '21-50' if n <= 50 else '51+'
    def longtail_ndcg(recs, test_items, train_items, k=10, buckets=('2-5', '6-20', '21-50', '51+')):
        def _u_nr(rec, rel, kk):
            hits = [1 if it in rel else 0 for it in rec[:kk]]; nh = sum(hits)
            dcg = sum(1 / math.log2(i + 2) for i, h in enumerate(hits) if h)
            idcg = sum(1 / math.log2(i + 2) for i in range(min(len(rel), kk)))
            return (dcg / idcg if idcg > 0 else 0.0, nh / len(rel) if rel else 0.0)
        act = {u: _bucket_f2(len(train_items.get(u, set()))) for u in recs}; out = {}
        for b in buckets:
            us = [u for u in recs if act.get(u) == b and test_items.get(u)]
            if not us: continue
            out[b] = {'n': len(us),
                      f'NDCG@{k}': float(np.mean([_u_nr(recs[u], test_items[u], k)[0] for u in us])),
                      f'Recall@{k}': float(np.mean([_u_nr(recs[u], test_items[u], k)[1] for u in us]))}
        return out
print('bootstrap OK | numpy', np.__version__, '| pandas', pd.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 93.6 MB/s eta 0:00:00
[protocol] v2026-06-23b (defaults: eval sin tope max_eval_users=None + frac_train=0.30)
[protocol] seed global = 42 | numpy/random/torch (cuda=True, determinista=True)
[F2] recsys_protocol clonado sin sección F2 -> usando definiciones inline (idénticas al módulo)
bootstrap OK | numpy 2.0.2 | pandas 2.2.2


In [2]:
# ====== Config — SCGRec (re-split RANDOM 80/10/10 de inter_train, full-ranking) ======
DATASET = 'scgrec'
MODEL_NAME = 'BPR'
KS = (5, 10)
ALS_FACTORS, ALS_ITERS, ALS_REG, ALS_ALPHA = 64, 15, 0.1, 40.0
N_EVAL_ANALYSIS = 200_000
TIER = 'T2'
SUBSAMPLE_USERS = 2_000_000 if TIER == 'T2' else 200_000   # = corregido T2 (mismo eval_set -> self-check)
N_EXAMPLES = 3
# Cache de SCGRec (load_scgrec -> scgrec_ready/). AUTOCONTENIDO: si faltan, se descargan+parsean
# desde el steam_data oficial (gdown). Usa Drive si está montado; si no, disco local efímero.
_drive = '/content/drive/MyDrive'
SCG_DIR = os.environ.get('SCG_DIR') or (f'{_drive}/scgrec_ready' if os.path.isdir(_drive) else 'scgrec_ready')
os.makedirs(SCG_DIR, exist_ok=True)
OUT = f'bpr_scgrec_h3'; os.makedirs(OUT, exist_ok=True)
print(f'BPR | SCGRec | TIER={TIER} | re-split 80/10/10 | subsample={SUBSAMPLE_USERS} | SCG_DIR={SCG_DIR}')


BPR | SCGRec | TIER=T2 | re-split 80/10/10 | subsample=2000000 | SCG_DIR=scgrec_ready


In [3]:
# ====== Carga SCGRec: bootstrap (gdown+parse si faltan) + re-split RANDOM 80/10/10 (= corregido T2) ======
# Mismo loader que load_scgrec.ipynb + misma carga que CPGRec_comparativo_multiseed_T2_corregido:
# F2 sobre SCGRec NO usa el split oficial valid/test -> toma inter_train (Table 1, 95,2M) y lo
# re-particiona RANDOM 80/10/10 (CPGRec §5.1.1). Así las filas BPR caen junto a MostPop/ALS/CPGRec.
# Bootstrap autocontenido: si faltan los parquets, baja+parsea el steam_data oficial (gdown, ~1GB, 1 vez).
SCGREC_ID='1F9kr_YWimBtexJEH-zkDzCOwl1q7GmFp'   # nota al pie 8 del paper SCGRec
_need=[f'{SCG_DIR}/inter_train.parquet', f'{SCG_DIR}/game_categories.parquet']
if all(os.path.exists(p) for p in _need):
    print('[scgrec] parquets presentes:', _need)
else:
    print('[scgrec] faltan parquets -> descargo+parseo steam_data oficial (gdown, ~1GB, 1 vez por sesión)')
    subprocess.run([sys.executable,'-m','pip','install','-q','-U','gdown'], check=False)  # NO subir pyarrow (rompe ABI en Colab)
    import gdown, tarfile
    import pyarrow as pa, pyarrow.parquet as pq
    from array import array
    from itertools import zip_longest
    _DATA='scgrec_data/steam_data'
    if not os.path.isdir(_DATA):
        _raw=gdown.download(id=SCGREC_ID, output='scgrec_raw', quiet=False)
        if zipfile.is_zipfile(_raw): zipfile.ZipFile(_raw).extractall('scgrec_data')
        elif tarfile.is_tarfile(_raw): tarfile.open(_raw).extractall('scgrec_data')
    _P=lambda *a: os.path.join(_DATA,*a)
    # inter_train.parquet: adyacencia (coma-sep, user_id al inicio, tiempos con \\N) -> streaming a parquet
    _SCHEMA=pa.schema([('user_id',pa.int64()),('app_id',pa.int32()),('playtime',pa.float32())])
    _games=set(); _wr=pq.ParquetWriter(_need[0],_SCHEMA)
    _bu,_bg,_bt=array('q'),array('i'),array('f')
    def _flush():
        global _bu,_bg,_bt
        if len(_bu):
            _wr.write_table(pa.table({'user_id':pa.array(_bu,pa.int64()),'app_id':pa.array(_bg,pa.int32()),'playtime':pa.array(_bt,pa.float32())}))
            _bu,_bg,_bt=array('q'),array('i'),array('f')
    with open(_P('train_game.txt')) as _fg, open(_P('train_time.txt')) as _ft:
        for _lg,_lt in zip(_fg,_ft):
            _tg=_lg.rstrip('\n').split(','); _tt=_lt.rstrip('\n').split(',')
            if not _tg or _tg[0]=='': continue
            _uid=int(_tg[0])
            for _g,_t in zip_longest(_tg[1:],_tt[1:],fillvalue='\\N'):
                if _g is None or _g=='': continue
                _gi=int(_g); _games.add(_gi); _bu.append(_uid); _bg.append(_gi)
                _bt.append(0.0 if _t in (None,'\\N','') else float(_t))
            if len(_bu)>=4_000_000: _flush()
    _flush(); _wr.close()
    # game_categories.parquet: pares app_id,valor (partition por 1ª coma) + App_ID_Info
    def _pairs(path,col):
        a,v=[],[]
        for line in open(path,encoding='utf-8'):
            line=line.rstrip('\n')
            if line: k,_,val=line.partition(','); a.append(int(k)); v.append(val)
        return pd.DataFrame({'app_id':a,col:v})
    _g2=_pairs(_P('Games_Genres.txt'),'genre').groupby('app_id')['genre'].apply(list)
    _d2=_pairs(_P('Games_Developers.txt'),'developer').groupby('app_id')['developer'].apply(list)
    _p2=_pairs(_P('Games_Publishers.txt'),'publisher').groupby('app_id')['publisher'].apply(list)
    _ai=pd.read_csv(_P('App_ID_Info.txt'),header=None,
                    names=['app_id','name','type','price','release_date','metascore','c6','c7'],
                    on_bad_lines='skip',engine='python')
    _allg=pd.Index(sorted(_games|set(_ai['app_id'])))
    _cat=pd.DataFrame({'app_id':_allg})
    _cat['genres']=_cat['app_id'].map(lambda x:_g2.get(x,[])); _cat['developers']=_cat['app_id'].map(lambda x:_d2.get(x,[]))
    _cat['publishers']=_cat['app_id'].map(lambda x:_p2.get(x,[]))
    _cat=_cat.merge(_ai[['app_id','name','type','price','release_date','metascore']],on='app_id',how='left')
    _cat.to_parquet(_need[1],index=False); print('[scgrec] parquets listos | juegos:',len(_allg))

inter=pd.read_parquet(_need[0])
inter['user_id']=inter['user_id'].astype('int64'); inter['app_id']=inter['app_id'].astype('int64')
if SUBSAMPLE_USERS is not None:
    _rng=np.random.default_rng(SEED); _u=np.sort(inter['user_id'].unique())
    keep=set(_rng.choice(_u, size=min(SUBSAMPLE_USERS,len(_u)), replace=False).tolist())
    inter=inter[inter['user_id'].isin(keep)].copy()
inter=random_split_8010(inter, SEED)   # re-split 80/10/10 (idéntico al corregido)
print(f'inter(re-split 80/10/10)={len(inter):,} | usuarios={inter["user_id"].nunique():,} | '
      f'juegos={inter["app_id"].nunique():,} | playtime=0: {inter["playtime"].eq(0).mean()*100:.0f}%')


[scgrec] faltan parquets -> descargo+parseo steam_data oficial (gdown, ~1GB, 1 vez por sesión)


Downloading...
From (original): https://drive.google.com/uc?id=1F9kr_YWimBtexJEH-zkDzCOwl1q7GmFp
From (redirected): https://drive.google.com/uc?id=1F9kr_YWimBtexJEH-zkDzCOwl1q7GmFp&confirm=t&uuid=fa796134-6914-4c83-b344-efa2cd74a47c
To: /content/scgrec_raw
100%|██████████| 993M/993M [00:21<00:00, 45.6MB/s]


[scgrec] parquets listos | juegos: 2675
inter(re-split 80/10/10)=48,697,955 | usuarios=2,000,000 | juegos=2,650 | playtime=0: 40%


In [4]:
# ====== Categorias SCGRec (game_categories.parquet; CATALOG = catálogo completo; SIN texto -> solo BPR) ======
# Igual que el corregido T2: CATALOG sale de game_categories (todos los juegos), no de inter.
cat=pd.read_parquet(f'{SCG_DIR}/game_categories.parquet')
cat['app_id']=pd.to_numeric(cat['app_id'],errors='coerce')
cat=cat.dropna(subset=['app_id']); cat['app_id']=cat['app_id'].astype('int64'); cat=cat.drop_duplicates('app_id').reset_index(drop=True)
for _c in ('genres','developers','publishers'):
    if _c not in cat.columns: cat[_c]=[[] for _ in range(len(cat))]
cat['name']=[(str(n) if isinstance(n,str) and str(n).strip() else f'app_{a}')
             for a,n in zip(cat['app_id'], (cat['name'] if 'name' in cat.columns else [None]*len(cat)))]
cat['description']=''
CATALOG=sorted(int(a) for a in cat['app_id'].unique()); catalog_set=set(CATALOG); n_catalog=len(CATALOG)
desc_map={a:'' for a in CATALOG}; tags_map={a:[] for a in CATALOG}
_cov=float((cat['genres'].map(lambda v: len(v) if hasattr(v,'__len__') else 0)>0).mean())
print(f'cat={len(cat):,} juegos | cobertura géneros={_cov:.2f} (SCGRec ~0.75)')


cat=2,675 juegos | cobertura géneros=0.75 (SCGRec ~0.75)


In [5]:
# ====== Mapas de categoria + train/test + por-usuario (comun) ======
genre_map={int(a):list(g) for a,g in zip(cat['app_id'],cat['genres'])}
dev_map  ={int(a):list(g) for a,g in zip(cat['app_id'],cat['developers'])}
pub_map  ={int(a):list(g) for a,g in zip(cat['app_id'],cat['publishers'])}
total_map={a:[('g',x) for x in genre_map.get(a,[])]+[('d',x) for x in dev_map.get(a,[])]
              +[('p',x) for x in pub_map.get(a,[])] for a in CATALOG}
CAT_MAPS={'gene':genre_map,'dev':dev_map,'pub':pub_map,'total':total_map}
name_map={int(a):n for a,n in zip(cat['app_id'],cat['name'])}

train=inter[inter['split']==0]; test=inter[inter['split']==2]
train_items_per_user=train.groupby('user_id')['app_id'].apply(lambda s:set(int(x) for x in s)).to_dict()
test_items_per_user ={u:set(int(x) for x in g) for u,g in test.groupby('user_id')['app_id']}
eval_users=[u for u in test_items_per_user if u in train_items_per_user]
print(f'train={len(train):,} test={len(test):,} | eval usuarios={len(eval_users):,} | '
      f'positivos/usuario(medio)={np.mean([len(test_items_per_user[u]) for u in eval_users]):.2f}')


train=38,958,364 test=4,869,796 | eval usuarios=1,447,725 | positivos/usuario(medio)=3.36


In [6]:
# ====== MostPop + ALS (self-check) + eval_set fijo ======
import scipy.sparse as _sp
from implicit.als import AlternatingLeastSquares
pop=train.groupby('app_id').size().to_dict()
popular_list=[it for it,_ in sorted(pop.items(), key=lambda kv:(-kv[1],kv[0])) if it in catalog_set]
TOPN=max(KS)

als_users=sorted(train['user_id'].unique().tolist())
_u2i={u:i for i,u in enumerate(als_users)}; _a2i={a:i for i,a in enumerate(CATALOG)}
idx2app_als={i:a for a,i in _a2i.items()}
_tr=train[train['app_id'].isin(_a2i)]
_rows=_tr['user_id'].map(_u2i).to_numpy(); _cols=_tr['app_id'].map(_a2i).to_numpy()
_conf=(1.0+ALS_ALPHA*np.log1p(_tr['playtime'].fillna(0).clip(lower=0).to_numpy())).astype('float32')
_ui=_sp.csr_matrix((_conf,(_rows,_cols)), shape=(len(als_users),len(CATALOG)))
als=AlternatingLeastSquares(factors=ALS_FACTORS, regularization=ALS_REG, iterations=ALS_ITERS, random_state=SEED, use_gpu=False)
als.fit(_ui)
als_uf=np.asarray(als.user_factors); als_if=np.asarray(als.item_factors)
als_umap={str(u):_u2i[u] for u in als_users}
print('ALS listo:', als_uf.shape, als_if.shape)

# eval_set fijo (mismo criterio que el *_corregido: seed 42, tope N_EVAL_ANALYSIS)
if TIER=='T1': eval_users=eval_users[:2000]
if N_EVAL_ANALYSIS and len(eval_users)>N_EVAL_ANALYSIS:
    _rng=np.random.default_rng(SEED)
    eval_set=sorted(_rng.choice(np.array(eval_users), size=N_EVAL_ANALYSIS, replace=False).tolist())
else:
    eval_set=list(eval_users)
print(f'eval_set: {len(eval_set):,} usuarios')

recs_fixed={}
recs_fixed['Most Popular']={u:[i for i in popular_list if i not in train_items_per_user.get(u,set())][:TOPN] for u in eval_set}
recs_fixed['ALS'],_=recs_from_embeddings(als_uf,als_if,als_umap,idx2app_als,eval_set,TOPN,train_items_per_user,popular_list)
metrics_fixed={m:paper_metrics(recs_fixed[m],test_items_per_user,ks=KS,cat_maps=CAT_MAPS) for m in recs_fixed}
for m in recs_fixed:
    d=metrics_fixed[m]
    print(f'  {m:13s} R@5={d["Recall@5"]:.4f} NDCG@5={d["NDCG@5"]:.4f} NDCG@10={d["NDCG@10"]:.4f} '
          f'Hit@10={d["Hit@10"]:.4f} (self-check vs *_corregido)')


  0%|          | 0/15 [00:00<?, ?it/s]

ALS listo: (1999978, 64) (2675, 64)
eval_set: 200,000 usuarios
  Most Popular  R@5=0.2128 NDCG@5=0.1728 NDCG@10=0.2068 Hit@10=0.4670 (self-check vs *_corregido)
  ALS           R@5=0.3752 NDCG@5=0.3270 NDCG@10=0.3554 Hit@10=0.6375 (self-check vs *_corregido)


## Modelo: BPR

In [7]:
# ====== BPR (implicit) — matriz BINARIA, misma indexacion que ALS ======
try:
    from implicit.bpr import BayesianPersonalizedRanking
except ImportError:
    from implicit.cpu.bpr import BayesianPersonalizedRanking
_uib=_sp.csr_matrix((np.ones(len(_tr),dtype='float32'),(_rows,_cols)), shape=(len(als_users),len(CATALOG)))
bpr=BayesianPersonalizedRanking(factors=64, learning_rate=0.01, regularization=0.01, iterations=100, random_state=SEED)
bpr.fit(_uib)
bpr_uf=np.asarray(bpr.user_factors); bpr_if=np.asarray(bpr.item_factors)   # (N,F+1)/(M,F+1): bias plegado
recs_fixed['BPR'],_nfb=recs_from_embeddings(bpr_uf,bpr_if,als_umap,idx2app_als,eval_set,TOPN,train_items_per_user,popular_list)
metrics_fixed['BPR']=paper_metrics(recs_fixed['BPR'],test_items_per_user,ks=KS,cat_maps=CAT_MAPS)
d=metrics_fixed['BPR']
print(f'BPR  R@5={d["Recall@5"]:.4f} NDCG@5={d["NDCG@5"]:.4f} NDCG@10={d["NDCG@10"]:.4f} '
      f'Hit@10={d["Hit@10"]:.4f} | cold-fallback={_nfb}')


  0%|          | 0/100 [00:00<?, ?it/s]

BPR  R@5=0.4145 NDCG@5=0.3725 NDCG@10=0.3968 Hit@10=0.6758 | cold-fallback=0


## Análisis (A accuracy · B diversidad · C long-tail · D ejemplos)

In [8]:
# ====== A/B/C/D (accuracy, diversidad, long-tail, ejemplos) + guardar + imprimir ======
import json as _json
MODELS=['Most Popular','ALS',MODEL_NAME]
_MET=['Recall@5','NDCG@5','Hit@5','Precision@5','Recall@10','NDCG@10']
dfA=pd.DataFrame([[m]+[f'{metrics_fixed[m][x]:.4f}' for x in _MET] for m in MODELS], columns=['Modelo']+_MET)
print('=== A. Accuracy (split 80/10/10, full-ranking, 1 seed) ===')
print(dfA.to_string(index=False))

DIV=[f'Cov_total@{k}' for k in KS]+[f'Cov_gene@{k}' for k in KS]+[f'Ent_gene@{k}' for k in KS]
dfB=pd.DataFrame({m:{c:f'{metrics_fixed[m][c]:.4f}' for c in DIV} for m in MODELS}).T[DIV]
print('\n=== B. Diversidad (Cov=nro categorias distintas en top-K; Ent=entropia) ===')
print(dfB.to_string())

lt={m:longtail_ndcg(recs_fixed[m],test_items_per_user,train_items_per_user,k=10) for m in MODELS}
buckets=['2-5','6-20','21-50','51+']
rowsC=[]
for b in buckets:
    row={'actividad':b}
    for m in MODELS: row[m]=(f'{lt[m][b]["NDCG@10"]:.4f}' if b in lt[m] else '')
    rowsC.append(row)
dfC=pd.DataFrame(rowsC)[['actividad']+MODELS]
print('\n=== C. Long-tail: NDCG@10 por actividad del usuario ===')
print(dfC.to_string(index=False))
print('n usuarios/bucket:', {b:(lt['Most Popular'][b]['n'] if b in lt['Most Popular'] else 0) for b in buckets})

def _u_nr5(rec,rel):
    hits=[1 if it in rel else 0 for it in rec[:5]]; nh=sum(hits)
    dcg=sum(1/math.log2(i+2) for i,h in enumerate(hits) if h)
    idcg=sum(1/math.log2(i+2) for i in range(min(len(rel),5)))
    return (dcg/idcg if idcg>0 else 0.0, nh/len(rel) if rel else 0.0)
_act={u:('2-5' if len(train_items_per_user.get(u,set()))<=5 else '6-20' if len(train_items_per_user.get(u,set()))<=20
         else '21-50' if len(train_items_per_user.get(u,set()))<=50 else '51+') for u in eval_set}
ej=[f'(ejemplos seed {SEED})']; _chosen=[]
for b in buckets:
    for u in [x for x in eval_set if _act[x]==b and test_items_per_user.get(x)]:
        if all(u in recs_fixed[m] for m in MODELS) and any(any(it in test_items_per_user[u] for it in recs_fixed[m][u][:5]) for m in MODELS):
            _chosen.append((b,u)); break
    if len(_chosen)>=N_EXAMPLES: break
def _nm(items): return [name_map.get(a,str(a)) for a in items]
for b,u in _chosen[:N_EXAMPLES]:
    rel=test_items_per_user[u]; hist=sorted(train_items_per_user.get(u,set()))
    ej.append(f'\n### Usuario {u} (actividad {b}, {len(hist)} juegos en historial)')
    ej.append('- Perfil (muestra): '+', '.join(_nm(hist[:6])))
    ej.append('- Test (a acertar): '+', '.join(_nm(sorted(rel))))
    for m in MODELS:
        nd,rc=_u_nr5(recs_fixed[m][u],rel)
        marks=[name_map.get(a,str(a))+(' ✓' if a in rel else '') for a in recs_fixed[m][u][:5]]
        ej.append(f'  - **{m}** (R@5={rc:.2f}, NDCG@5={nd:.2f}): '+', '.join(marks))
ej_txt='\n'.join(ej)
print('\n=== D. Ejemplos ===\n'+ej_txt)

dfA.to_csv(f'{OUT}/accuracy.csv', index=False); dfB.to_csv(f'{OUT}/diversidad.csv'); dfC.to_csv(f'{OUT}/longtail.csv', index=False)
open(f'{OUT}/ejemplos.md','w',encoding='utf-8').write(ej_txt)
_full={'dataset':DATASET,'model':MODEL_NAME,'seed':SEED,'tier':TIER,'n_eval':len(eval_set),
       'metrics':{m:{k:(float(v) if isinstance(v,(int,float,np.floating)) else v) for k,v in metrics_fixed[m].items()} for m in MODELS}}
_json.dump(_full, open(f'{OUT}/metricas_full.json','w'), indent=2)
print('\n'+'#'*72+'\n# RESPALDO EN TEXTO (todo impreso por si no se descargan archivos)\n'+'#'*72)
print('\n== accuracy.csv ==\n'+dfA.to_csv(index=False))
print('== diversidad.csv ==\n'+dfB.to_csv())
print('== longtail.csv ==\n'+dfC.to_csv(index=False))
print('== metricas_full.json ==\n'+_json.dumps(_full, indent=2))
try:
    shutil.make_archive(OUT,'zip',OUT)
    from google.colab import files; files.download(OUT+'.zip'); print('(zip de descarga generado)')
except Exception as e:
    print('(descarga automatica no disponible:', repr(e), '-> todo esta impreso arriba)')


=== A. Accuracy (split 80/10/10, full-ranking, 1 seed) ===
      Modelo Recall@5 NDCG@5  Hit@5 Precision@5 Recall@10 NDCG@10
Most Popular   0.2128 0.1728 0.3494      0.0895    0.3104  0.2068
         ALS   0.3752 0.3270 0.5298      0.1369    0.4520  0.3554
         BPR   0.4145 0.3725 0.5897      0.1634    0.4885  0.3968

=== B. Diversidad (Cov=nro categorias distintas en top-K; Ent=entropia) ===
             Cov_total@5 Cov_total@10 Cov_gene@5 Cov_gene@10 Ent_gene@5 Ent_gene@10
Most Popular      3.1342       4.1366     1.0915      1.6854     0.0557      0.3081
ALS              11.0786      19.9667     3.3178      4.8900     1.3362      1.7669
BPR               8.6660      14.5117     2.6583      3.7683     0.9552      1.2875

=== C. Long-tail: NDCG@10 por actividad del usuario ===
actividad Most Popular    ALS    BPR
      2-5       0.2739 0.5933 0.6271
     6-20       0.2196 0.4000 0.4309
    21-50       0.1566 0.1751 0.2281
      51+       0.1330 0.0996 0.1795
n usuarios/bucket: {'2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

(zip de descarga generado)
